# Exploration

Ce notebook présente la structure de base de l'entraînement du modèle.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

from training.data import load_data

In [2]:
# On utilise la fonction load_data qui standardise le chargement des données pour éviter de répéter le code dans chaque notebook
df = load_data('../data/employees.csv')

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import f1_score, roc_auc_score

target = "LeaveOrNot"
X = df.drop(columns=[target])
y = df[target]

categorical_features = X.select_dtypes(include=["object", "string"]).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), categorical_features),
    ],
    remainder="passthrough",
)
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42)),
])

param_grid = {
    "classifier__n_estimators": [50, 100, 200],
    "classifier__max_depth": [5, 10],
}

grid_search = GridSearchCV(
    pipeline, param_grid, cv=5, scoring="accuracy", n_jobs=-1
)
grid_search.fit(X_train, y_train)

test_accuracy = grid_search.score(X_test, y_test)
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV accuracy: {grid_search.best_score_:.4f}")
print(f"Test accuracy:    {test_accuracy:.4f}")

Best parameters: {'classifier__max_depth': 10, 'classifier__n_estimators': 200}
Best CV accuracy: 0.8436
Test accuracy:    0.8700


In [4]:
import mlflow, joblib

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("employee_attrition_notebook")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1782277653229, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1782277653229, lifecycle_stage='active', name='employee_attrition_notebook', tags={}, trace_location=None, workspace='default'>

In [ ]:
with mlflow.start_run():
    grid_search.fit(X_train, y_train)
    mlflow.log_metric("test_f1", f1_score(y_test, grid_search.predict(X_test)))
    mlflow.log_metric("test_auc", roc_auc_score(y_test, grid_search.predict_proba(X_test)[:, 1]))
    joblib.dump(grid_search, "../models/employee_attrition_model.joblib")

🏃 View run judicious-wolf-723 at: http://127.0.0.1:5000/#/experiments/1/runs/a9d0c19028db4d1d95c4c20ad8c3a1c5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [6]:
import mlflow.sklearn

mlflow.sklearn.autolog()
with mlflow.start_run():
    grid_search.fit(X_train, y_train)
    mlflow.log_metric("test_f1", f1_score(y_test, grid_search.predict(X_test)))
    mlflow.log_metric("test_auc", roc_auc_score(y_test, grid_search.predict_proba(X_test)[:, 1]))
    joblib.dump(grid_search, "../models/employee_attrition_model.joblib")

2026/06/24 07:24:53 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\MIGNOT\Documents\Formation\MLOps\mlops-training\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/06/24 07:24:55 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\MIGNOT\D

🏃 View run amusing-shrew-299 at: http://127.0.0.1:5000/#/experiments/1/runs/43810f2d297546f3a2f345faed000fc3
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run nervous-fowl-386 at: http://127.0.0.1:5000/#/experiments/1/runs/f7ba53346ca045158cabfc1278eb1332
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run caring-grouse-683 at: http://127.0.0.1:5000/#/experiments/1/runs/2f8426dc20974f91b511682d12b6f6a8
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run capable-zebra-467 at: http://127.0.0.1:5000/#/experiments/1/runs/26f397390fed492b83a5a1ec4da57aef
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/06/24 07:25:10 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\MIGNOT\Documents\Formation\MLOps\mlops-training\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/06/24 07:25:10 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\MIGNOT\D

🏃 View run rebellious-slug-316 at: http://127.0.0.1:5000/#/experiments/1/runs/77ddec6a524a4f2cb37ba8fd3241a62d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run defiant-grub-302 at: http://127.0.0.1:5000/#/experiments/1/runs/a0eaa83b6be74c21bbc44d8375a7ff54
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
